## Демо 4 - Читаем WAL из слота прямо в SQL

В демо 2 и 3 изменения из Postgres доставал Debezium.<br>
Здесь сделаем то же самое руками, без Kafka и коннекторов: создадим логический слот и почитаем из него обычными `select`.

```bash
docker-compose up -d
```

Postgres: `localhost:5432`, логин и пароль `postgres`.<br>
SQL выполняем отсюда, через функции `sql()` и `show()`.

In [ ]:
import json
import struct

import psycopg2

conn = psycopg2.connect(host="localhost", port=5432, user="postgres", password="postgres", dbname="postgres")
conn.autocommit = True


def sql(query, params=None):
    # Просто выполняем запрос
    with conn.cursor() as cur:
        cur.execute(query, params)


def show(query, params=None):
    # Выполняем запрос и печатаем результат таблицей
    with conn.cursor() as cur:
        cur.execute(query, params)
        header = [column.name for column in cur.description]
        rows = cur.fetchall()
    widths = [max(len(str(x)) for x in column) for column in zip(header, *rows)]
    for row in [header] + rows:
        print("  ".join(str(x).ljust(w) for x, w in zip(row, widths)))
    print(f"строк: {len(rows)}")

### Шаг 1. Настройки

Логические слоты работают только при `wal_level = logical`. В `docker-compose.yaml` мы это включили.

In [ ]:
show("show wal_level")

In [ ]:
# Чистим за прошлым запуском, чтобы ноутбук можно было гонять повторно
sql("""
select pg_drop_replication_slot(slot_name) from pg_replication_slots
where slot_name in ('demo_slot', 'demo_pgoutput');
drop publication if exists demo_pub;
drop table if exists public.customers, public.big;
""")

### Шаг 2. Создаем слот

Слот - это закладка в WAL: Postgres запоминает, докуда мы дочитали, и не удаляет WAL, который мы еще не прочитали.<br>
Вместе со слотом выбираем плагин, который переводит WAL в понятный вид. `test_decoding` - учебный, он выдает текст.

Слот видит только то, что произошло после его создания, поэтому создаем его до изменений.

In [ ]:
show("select * from pg_create_logical_replication_slot('demo_slot', 'test_decoding')")

In [ ]:
show("select slot_name, plugin, slot_type, active, restart_lsn, confirmed_flush_lsn from pg_replication_slots")

### Шаг 3. Меняем данные и читаем слот

In [ ]:
sql("create table public.customers (id int primary key, name text)")

# Один insert на две строки - это одна транзакция
sql("insert into public.customers values (1, 'ilya'), (2, 'kate')")
sql("update public.customers set name = 'ilya2' where id = 1")
sql("delete from public.customers where id = 2")

In [ ]:
# peek - только посмотреть, слот при этом не двигается
show("select lsn, xid, data from pg_logical_slot_peek_changes('demo_slot', null, null)")

Что тут видно:
1) Изменения приходят по транзакциям: `BEGIN`, строки, `COMMIT`
2) Первая пара `BEGIN` / `COMMIT` пустая - это наш `create table`. DDL в поток не попадает
3) Один `insert` на две строки - это две строки `INSERT` в одной транзакции
4) В `UPDATE` только новая строка, а в `DELETE` только `id`. Старых значений нет

### Шаг 4. peek и get

`peek` только смотрит, поэтому его можно вызывать сколько угодно раз.<br>
`get` забирает изменения: слот двигается вперед, и прочитанное второй раз уже не отдается.<br>
Это как закоммиченный offset в Kafka.

In [ ]:
show("select count(*) as peek_1 from pg_logical_slot_peek_changes('demo_slot', null, null)")
show("select count(*) as peek_2 from pg_logical_slot_peek_changes('demo_slot', null, null)")
show("select count(*) as get_1 from pg_logical_slot_get_changes('demo_slot', null, null)")
show("select count(*) as get_2 from pg_logical_slot_get_changes('demo_slot', null, null)")

### Шаг 5. REPLICA IDENTITY FULL

Помните `replica identity full` из демо 2? Включим и посмотрим, что изменится.

In [ ]:
sql("alter table public.customers replica identity full")

sql("update public.customers set name = 'ilya3' where id = 1")
sql("delete from public.customers where id = 1")

# Опции плагина: без номеров транзакций и без пустых транзакций, чтобы вывод был короче
show("""
select data from pg_logical_slot_get_changes('demo_slot', null, null,
    'include-xids', '0', 'skip-empty-xacts', '1')
""")

Теперь в `UPDATE` есть `old-key` (старая строка) и `new-tuple` (новая), а в `DELETE` вся удаленная строка.<br>
Это те самые `before` и `after`, которые мы видели в сообщениях Debezium.

### Шаг 6. Транзакции

В слот попадают только закоммиченные транзакции, и в порядке коммита.<br>
Откроем транзакцию в отдельном соединении и не будем ее закрывать.

In [ ]:
OPTIONS = "'include-xids', '0', 'skip-empty-xacts', '1'"

other = psycopg2.connect(host="localhost", port=5432, user="postgres", password="postgres", dbname="postgres")
cur = other.cursor()  # autocommit тут выключен, значит это транзакция

cur.execute("insert into public.customers values (10, 'in-flight')")

show(f"select data from pg_logical_slot_peek_changes('demo_slot', null, null, {OPTIONS})")

Пусто: строка уже вставлена, но транзакция не закрыта.

In [ ]:
other.commit()

show(f"select data from pg_logical_slot_get_changes('demo_slot', null, null, {OPTIONS})")

После коммита строка появилась. А теперь откат:

In [ ]:
cur.execute("insert into public.customers values (11, 'ghost')")
other.rollback()

show(f"select data from pg_logical_slot_get_changes('demo_slot', null, null, {OPTIONS})")

### Шаг 7. Слот держит WAL

Пока слот не дочитан, Postgres не может удалить WAL, который тот еще не прочитал.<br>
Смотрим, сколько WAL держит слот (`wal_held`):

In [ ]:
SLOT_LAG = """
select slot_name, active, wal_status,
       pg_size_pretty(pg_wal_lsn_diff(pg_current_wal_lsn(), restart_lsn)) as wal_held
from pg_replication_slots
"""

show(SLOT_LAG)

Теперь нагенерируем изменений и ничего не будем читать:

In [ ]:
sql("create table public.big as select g as id, repeat('x', 100) as payload from generate_series(1, 200000) g")

show(SLOT_LAG)

Слот держит все, что с тех пор записал Postgres. Теперь дочитаем.<br>
Позиция слота сдвигается не сразу, а когда он встретит следующую служебную запись WAL, поэтому делаем `checkpoint` и читаем еще раз.

In [ ]:
show("select count(*) as consumed from pg_logical_slot_get_changes('demo_slot', null, null)")

sql("checkpoint")
show("select count(*) as consumed from pg_logical_slot_get_changes('demo_slot', null, null)")

show(SLOT_LAG)

Если читателя нет (Debezium остановили или про слот забыли), WAL копится и может занять весь диск.<br>
Поэтому слоты, которые не нужны, удаляют. А начиная с Postgres 13 рост можно ограничить параметром `max_slot_wal_keep_size`.

### Шаг 8. pgoutput: то, что читает Debezium

Debezium использует не `test_decoding`, а плагин `pgoutput`. Он отдает не текст, а компактный бинарный протокол.<br>
Читается так же, через SQL, только функция другая: `pg_logical_slot_peek_binary_changes`.

`pgoutput` отдает только таблицы из публикации, поэтому сначала создаем ее.

In [ ]:
sql("create publication demo_pub for table public.customers")
show("select * from pg_create_logical_replication_slot('demo_pgoutput', 'pgoutput')")

sql("insert into public.customers values (7, 'vova')")
sql("update public.customers set name = 'vova2' where id = 7")
sql("delete from public.customers where id = 7")

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        select lsn, data from pg_logical_slot_peek_binary_changes('demo_pgoutput', null, null,
            'proto_version', '1', 'publication_names', 'demo_pub')
    """)
    messages = cur.fetchall()

for lsn, data in messages:
    print(lsn, len(data), "байт", bytes(data)[:44])

Это байты. Первый символ - тип сообщения: `B` begin, `R` описание таблицы, `I` insert, `U` update, `D` delete, `C` commit.<br>
Чтобы получить из них строки, нужен парсер. Напишем маленький: ровно этим внутри занят Debezium.

In [ ]:
relations = {}  # id таблицы -> (имя таблицы, список колонок), приходит в сообщении R


def read_cstring(data, pos):
    end = data.index(b"\x00", pos)
    return data[pos:end].decode(), end + 1


def read_tuple(data, pos, columns):
    (count,) = struct.unpack_from("!h", data, pos)
    pos += 2
    row = {}
    for i in range(count):
        kind = chr(data[pos])
        pos += 1
        if kind == "t":  # значение пришло текстом: длина и сами байты
            (length,) = struct.unpack_from("!i", data, pos)
            pos += 4
            row[columns[i]] = data[pos:pos + length].decode()
            pos += length
        else:  # 'n' - NULL, 'u' - значение не менялось
            row[columns[i]] = None
    return row, pos


def decode(data):
    data = bytes(data)
    kind = chr(data[0])

    if kind == "B":
        (_, _, xid) = struct.unpack_from("!qqi", data, 1)
        return f"BEGIN xid={xid}"
    if kind == "C":
        return "COMMIT"
    if kind == "R":
        (relid,) = struct.unpack_from("!i", data, 1)
        schema, pos = read_cstring(data, 5)
        table, pos = read_cstring(data, pos)
        replica_identity = chr(data[pos])
        (count,) = struct.unpack_from("!h", data, pos + 1)
        pos += 3
        columns = []
        for _ in range(count):
            name, pos = read_cstring(data, pos + 1)  # +1: флаги колонки
            pos += 8  # тип колонки и typmod
            columns.append(name)
        relations[relid] = (f"{schema}.{table}", columns)
        return f"RELATION {schema}.{table} columns={columns} replica_identity={replica_identity}"
    if kind in ("I", "U", "D"):
        (relid,) = struct.unpack_from("!i", data, 1)
        table, columns = relations[relid]
        pos = 5
        before = after = None
        if kind != "I" and chr(data[pos]) in ("K", "O"):  # K - старый ключ, O - вся старая строка
            before, pos = read_tuple(data, pos + 1, columns)
        if kind != "D":  # дальше идет N - новая строка
            after, pos = read_tuple(data, pos + 1, columns)
        return {"op": {"I": "c", "U": "u", "D": "d"}[kind], "table": table, "before": before, "after": after}
    return f"тип сообщения {kind!r} пропускаем"


for lsn, data in messages:
    print(lsn, decode(data))

Получилось то же, что в сообщениях Debezium из демо 2: `op`, `before`, `after`.<br>
Остальное (JSON, типы колонок, отправка в Kafka) коннектор делает сверху. Слот Debezium из демо 2 тоже можно увидеть: `select * from pg_replication_slots`, там `dbname_debezium` с плагином `pgoutput`.

### Уборка

Слоты за собой надо удалять, иначе они продолжат держать WAL.

In [ ]:
other.close()

sql("""
select pg_drop_replication_slot('demo_slot');
select pg_drop_replication_slot('demo_pgoutput');
drop publication demo_pub;
drop table public.big;
""")

show("select count(*) as slots_left from pg_replication_slots")